# Notebook 27 — Residual Spectral Transition Analysis

This notebook extends the residual universality manifold paper with a true spectral-operator layer.

It generates Laplacian spectrum diagnostics for the residual manifold graph:

- residual Laplacian eigenspectrum,
- eigengap structure,
- spectral density and cumulative spectrum,
- eigenvalue spacing statistics,
- spectral entropy,
- topology-family spectral trajectories,
- spectral overlap matrix.

The notebook is designed to work even when `results/` is empty in Colab. If prior notebook outputs are unavailable, it regenerates a deterministic residual manifold baseline compatible with Notebooks 24–26.


In [ ]:
# Notebook 27 setup
from pathlib import Path
import json
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import pairwise_distances, cosine_similarity

try:
    import networkx as nx
except ImportError as e:
    raise ImportError('This notebook requires networkx. In Colab: !pip install networkx') from e

try:
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import laplacian
    from scipy.linalg import eigh
except ImportError as e:
    raise ImportError('This notebook requires scipy. In Colab: !pip install scipy') from e

# Resolve repo-style paths robustly for Colab and local execution.
CWD = Path.cwd()
if CWD.name == 'notebooks':
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = REPO_ROOT / 'figures'
EXPORTS_DIR = REPO_ROOT / 'exports'
for d in [RESULTS_DIR, FIGURES_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 9423
rng = np.random.default_rng(SEED)

print('cwd:', CWD)
print('repo root:', REPO_ROOT)
print('results:', RESULTS_DIR)
print('figures:', FIGURES_DIR)
print('exports:', EXPORTS_DIR)


## 1. Load existing residual manifold data or regenerate baseline

The notebook first looks for residual manifold coordinates from earlier notebooks. If none are present, it regenerates a deterministic graph-family dataset using the same family names and graph sizes used in the paper.


In [ ]:
def read_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f'loaded: {p}')
            return pd.read_csv(p), p
    return None, None

candidate_files = [
    RESULTS_DIR / 'residual_universality_embedding.csv',
    RESULTS_DIR / 'residual_pca_embedding.csv',
    RESULTS_DIR / 'residual_classification_feature_matrix.csv',
    RESULTS_DIR / 'residual_geometry_features.csv',
    RESULTS_DIR / '25_transport_nodes.csv',
    RESULTS_DIR / '26_diffusion_nodes.csv',
]

raw, source_path = read_first_existing(candidate_files)
print('available result files:', sorted(p.name for p in RESULTS_DIR.glob('*'))[:40])


In [ ]:
FAMILIES = ['ring lattice', 'small world', 'Erdős–Rényi', 'scale free', 'modular clustered']
SIZES = [16, 32, 64, 128]

def safe_graph_features(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    deg = np.array([d for _, d in G.degree()], dtype=float)
    if n == 0:
        raise ValueError('empty graph')
    density = nx.density(G)
    clustering = nx.average_clustering(G) if n > 1 else 0.0
    transitivity = nx.transitivity(G) if n > 2 else 0.0
    components = nx.number_connected_components(G)
    largest_cc = max(nx.connected_components(G), key=len)
    H = G.subgraph(largest_cc).copy()
    try:
        avg_path = nx.average_shortest_path_length(H) if H.number_of_nodes() > 1 else 0.0
    except Exception:
        avg_path = 0.0
    try:
        diameter = nx.diameter(H) if H.number_of_nodes() > 1 else 0.0
    except Exception:
        diameter = 0.0
    try:
        assort = nx.degree_assortativity_coefficient(G)
        if not np.isfinite(assort):
            assort = 0.0
    except Exception:
        assort = 0.0
    triangles = sum(nx.triangles(G).values()) / 3.0
    return {
        'n': n,
        'edges': m,
        'density': density,
        'mean_degree': float(deg.mean()),
        'std_degree': float(deg.std()),
        'max_degree': float(deg.max()),
        'degree_heterogeneity': float(deg.std() / (deg.mean() + 1e-9)),
        'clustering': clustering,
        'transitivity': transitivity,
        'components': components,
        'largest_component_frac': len(largest_cc) / n,
        'avg_path_largest_cc': avg_path,
        'diameter_largest_cc': diameter,
        'assortativity': assort,
        'triangles': triangles,
    }

def make_graph(family, n, seed):
    if family == 'ring lattice':
        k = min(4, n-1)
        if k % 2 == 1: k -= 1
        return nx.watts_strogatz_graph(n, k=max(k,2), p=0.0, seed=seed)
    if family == 'small world':
        k = min(4, n-1)
        if k % 2 == 1: k -= 1
        return nx.watts_strogatz_graph(n, k=max(k,2), p=0.18, seed=seed)
    if family == 'Erdős–Rényi':
        p = min(0.18, 4 / max(n-1, 1))
        return nx.erdos_renyi_graph(n, p=p, seed=seed)
    if family == 'scale free':
        m = max(1, min(3, n//10 + 1))
        return nx.barabasi_albert_graph(n, m=m, seed=seed)
    if family == 'modular clustered':
        sizes = [n//2, n - n//2]
        p_in = min(0.55, 6 / max(sizes[0], 2))
        p_out = 0.025
        return nx.stochastic_block_model(sizes, [[p_in, p_out], [p_out, p_in]], seed=seed)
    raise ValueError(f'unknown family: {family}')

def regenerate_baseline():
    rows = []
    graphs = {}
    for family in FAMILIES:
        for n in SIZES:
            seed = abs(hash((family, n, SEED))) % (2**32 - 1)
            G = make_graph(family, n, seed)
            graphs[(family, n)] = G
            row = safe_graph_features(G)
            row.update({'family': family, 'topology': family, 'graph_size': n, 'N': n})
            rows.append(row)
    df = pd.DataFrame(rows)
    feature_cols = [c for c in df.columns if c not in ['family','topology','graph_size','N']]
    X = StandardScaler().fit_transform(df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0))
    pca = PCA(n_components=2, random_state=SEED)
    coords = pca.fit_transform(X)
    df['PC1'] = coords[:,0]
    df['PC2'] = coords[:,1]
    df['source'] = 'regenerated_baseline_notebook_27'
    return df, graphs, feature_cols, pca.explained_variance_ratio_

if raw is None:
    print('No compatible residual manifold file found; regenerating deterministic baseline.')
    manifold_df, graph_store, feature_cols, pca_var = regenerate_baseline()
    source_path = RESULTS_DIR / '27_regenerated_residual_manifold.csv'
    manifold_df.to_csv(source_path, index=False)
else:
    manifold_df = raw.copy()
    # Normalize column names from earlier notebooks.
    rename_map = {}
    for c in manifold_df.columns:
        lc = c.lower()
        if lc in ['topology', 'label', 'graph_family']:
            rename_map[c] = 'family'
        if lc in ['n', 'size']:
            rename_map[c] = 'graph_size'
        if lc in ['x', 'pca1', 'coord1']:
            rename_map[c] = 'PC1'
        if lc in ['y', 'pca2', 'coord2']:
            rename_map[c] = 'PC2'
    manifold_df = manifold_df.rename(columns=rename_map)
    if 'family' not in manifold_df.columns and 'topology' in manifold_df.columns:
        manifold_df['family'] = manifold_df['topology']
    if 'graph_size' not in manifold_df.columns and 'N' in manifold_df.columns:
        manifold_df['graph_size'] = manifold_df['N']
    if 'N' not in manifold_df.columns and 'graph_size' in manifold_df.columns:
        manifold_df['N'] = manifold_df['graph_size']
    
    # If coordinates absent, project numeric features.
    if not {'PC1','PC2'}.issubset(manifold_df.columns):
        numeric_cols = manifold_df.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c not in ['N','graph_size']]
        X = StandardScaler().fit_transform(manifold_df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0))
        coords = PCA(n_components=2, random_state=SEED).fit_transform(X)
        manifold_df['PC1'] = coords[:,0]
        manifold_df['PC2'] = coords[:,1]
    
    # Build graph_store when possible from family/size for spectral trajectories.
    graph_store = {}
    for _, row in manifold_df.dropna(subset=['family','graph_size']).iterrows():
        family = row['family']
        n = int(row['graph_size'])
        if family in FAMILIES and n in SIZES:
            seed = abs(hash((family, n, SEED))) % (2**32 - 1)
            graph_store[(family, n)] = make_graph(family, n, seed)
    feature_cols = []
    pca_var = None

# Standardize family spelling for filenames/displays.
manifold_df['family'] = manifold_df['family'].astype(str)
manifold_df['graph_size'] = manifold_df['graph_size'].astype(int)
manifold_df['N'] = manifold_df['graph_size']

print('source_path:', source_path)
print(manifold_df[['family','graph_size','PC1','PC2']].head())
print('rows:', len(manifold_df))


## 2. Build residual manifold kNN graph and Laplacian spectrum

We construct a kNN graph over residual manifold coordinates and analyze the graph Laplacian:

\[
L = D - A
\]

with eigensystem:

\[
Lu_k = \lambda_k u_k.
\]


In [ ]:
coords = manifold_df[['PC1','PC2']].to_numpy(dtype=float)
n_nodes = len(coords)
k = min(4, max(2, n_nodes - 1))

nbrs = NearestNeighbors(n_neighbors=k+1).fit(coords)
distances, indices = nbrs.kneighbors(coords)

A = np.zeros((n_nodes, n_nodes), dtype=float)
for i in range(n_nodes):
    for dist, j in zip(distances[i,1:], indices[i,1:]):
        # Gaussian distance weight.
        sigma = np.median(distances[:,1:]) + 1e-9
        w = np.exp(-(dist**2)/(2*sigma**2))
        A[i,j] = max(A[i,j], w)
        A[j,i] = max(A[j,i], w)

D = np.diag(A.sum(axis=1))
L = D - A

# Numerical symmetry cleanup.
L = 0.5 * (L + L.T)
evals, evecs = eigh(L)
evals = np.maximum(evals, 0.0)

spectrum_df = pd.DataFrame({
    'mode': np.arange(len(evals)),
    'eigenvalue': evals,
})
spectrum_df['eigengap'] = spectrum_df['eigenvalue'].diff().fillna(0.0)
spectrum_df.to_csv(RESULTS_DIR / '27_laplacian_spectrum.csv', index=False)

# Attach first four nontrivial modes when available.
mode_df = manifold_df[['family','graph_size','PC1','PC2']].copy()
for idx, mode in enumerate(range(1, min(5, evecs.shape[1]))):
    mode_df[f'u{mode}'] = evecs[:, mode]
mode_df.to_csv(RESULTS_DIR / '27_laplacian_modes_by_node.csv', index=False)

print(spectrum_df.head(10))


## 3. Figure 1 — True residual Laplacian eigenspectrum

This figure replaces the previous placeholder/duplicate spectrum panel in the paper.


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(spectrum_df['mode'], spectrum_df['eigenvalue'], marker='o')
ax.set_title('Residual Laplacian eigenspectrum')
ax.set_xlabel('mode')
ax.set_ylabel('eigenvalue')
ax.grid(True, alpha=0.35)

# annotate largest nonzero eigengaps excluding first trivial jump when possible
if len(spectrum_df) > 4:
    gaps = spectrum_df.iloc[2:].copy()
    top = gaps.nlargest(min(3, len(gaps)), 'eigengap')
    for _, r in top.iterrows():
        ax.axvline(r['mode'], linestyle='--', alpha=0.35)
        ax.text(r['mode'], r['eigenvalue'], f"gap {r['eigengap']:.2f}", fontsize=9, ha='left', va='bottom')

fig.tight_layout()
out = FIGURES_DIR / '27_residual_laplacian_eigenspectrum.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print('wrote:', out)


## 4. Figure 2 — Spectral density and cumulative spectrum

The spectral density summarizes how eigenvalues concentrate across residual manifold modes.
The cumulative spectrum provides a compact continuity/fragmentation diagnostic.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].hist(evals, bins=min(12, max(5, len(evals)//2)), alpha=0.85)
axes[0].set_title('Residual spectral density')
axes[0].set_xlabel('eigenvalue')
axes[0].set_ylabel('count')
axes[0].grid(True, alpha=0.25)

cum = np.cumsum(evals) / (np.sum(evals) + 1e-12)
axes[1].plot(np.arange(len(cum)), cum, marker='o')
axes[1].set_title('Cumulative residual spectrum')
axes[1].set_xlabel('mode')
axes[1].set_ylabel('cumulative spectral mass')
axes[1].grid(True, alpha=0.25)

fig.tight_layout()
out = FIGURES_DIR / '27_spectral_density_and_cumulative.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print('wrote:', out)


## 5. Figure 3 — Eigenvalue spacing statistics

Eigenvalue spacing highlights transition regions between coarse residual organization and finer local variation.

\[
\Delta_k = \lambda_{k+1}-\lambda_k
\]


In [ ]:
spacing = np.diff(evals)
spacing_df = pd.DataFrame({'mode': np.arange(len(spacing)), 'spacing': spacing})
spacing_df.to_csv(RESULTS_DIR / '27_eigenvalue_spacing.csv', index=False)

fig, ax = plt.subplots(figsize=(10,6))
ax.bar(spacing_df['mode'], spacing_df['spacing'])
ax.set_title('Residual eigenvalue spacing')
ax.set_xlabel('mode k')
ax.set_ylabel(r'$\lambda_{k+1}-\lambda_k$')
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / '27_eigenvalue_spacing.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print('wrote:', out)


## 6. Figure 4 — Spectral entropy curve

Spectral entropy measures how Laplacian energy is distributed across modes.

\[
H_\lambda = -\sum_k p_k \log p_k,
\qquad
p_k = rac{\lambda_k}{\sum_j \lambda_j}
\]


In [ ]:
positive = evals[evals > 1e-12]
p = positive / (positive.sum() + 1e-12)
spectral_entropy = float(-np.sum(p * np.log(p + 1e-12)))

# Cumulative entropy as modes are included.
entropy_rows = []
for m in range(1, len(evals)+1):
    vals = evals[:m]
    vals = vals[vals > 1e-12]
    if len(vals) == 0:
        H = 0.0
    else:
        pm = vals / (vals.sum() + 1e-12)
        H = float(-np.sum(pm * np.log(pm + 1e-12)))
    entropy_rows.append({'mode_cutoff': m-1, 'spectral_entropy': H})

entropy_df = pd.DataFrame(entropy_rows)
entropy_df.to_csv(RESULTS_DIR / '27_spectral_entropy_curve.csv', index=False)

fig, ax = plt.subplots(figsize=(10,6))
ax.plot(entropy_df['mode_cutoff'], entropy_df['spectral_entropy'], marker='o')
ax.axhline(spectral_entropy, linestyle='--', alpha=0.5, label=f'full entropy = {spectral_entropy:.2f}')
ax.set_title('Residual spectral entropy curve')
ax.set_xlabel('mode cutoff')
ax.set_ylabel('spectral entropy')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / '27_spectral_entropy_curve.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print('wrote:', out)


## 7. Figure 5 — Topology-family spectral trajectories

For each generated graph family and size, compute a compact graph-level spectrum. These trajectories show how spectral summaries extend across graph size.


In [ ]:
def graph_spectral_summary(G, n_modes=8):
    A_g = nx.to_numpy_array(G, dtype=float)
    D_g = np.diag(A_g.sum(axis=1))
    L_g = 0.5*((D_g - A_g) + (D_g - A_g).T)
    vals = np.maximum(eigh(L_g, eigvals_only=True), 0.0)
    vals = np.sort(vals)
    pos = vals[vals > 1e-10]
    if len(pos) == 0:
        pos = np.array([0.0])
    gaps = np.diff(vals)
    return {
        'lambda_1': float(vals[1]) if len(vals) > 1 else 0.0,
        'lambda_2': float(vals[2]) if len(vals) > 2 else 0.0,
        'lambda_mean': float(vals.mean()),
        'lambda_max': float(vals.max()),
        'spectral_gap_max': float(gaps.max()) if len(gaps) else 0.0,
        'spectral_energy': float(np.sum(vals**2)),
        'spectral_entropy': float(-(lambda p: np.sum(p*np.log(p+1e-12)))(pos/(pos.sum()+1e-12))),
    }

spec_rows = []
for (family, n), G in sorted(graph_store.items(), key=lambda x: (str(x[0][0]), x[0][1])):
    row = graph_spectral_summary(G)
    row.update({'family': family, 'graph_size': n})
    spec_rows.append(row)

family_spec = pd.DataFrame(spec_rows)
family_spec.to_csv(RESULTS_DIR / '27_family_graph_spectral_summary.csv', index=False)

# PCA over graph-level spectral summaries.
num_cols = ['lambda_1','lambda_2','lambda_mean','lambda_max','spectral_gap_max','spectral_energy','spectral_entropy']
X_spec = StandardScaler().fit_transform(family_spec[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0))
spec_coords = PCA(n_components=2, random_state=SEED).fit_transform(X_spec)
family_spec['spectral_PC1'] = spec_coords[:,0]
family_spec['spectral_PC2'] = spec_coords[:,1]
family_spec.to_csv(RESULTS_DIR / '27_family_spectral_trajectories.csv', index=False)

fig, ax = plt.subplots(figsize=(11,8))
for family, g in family_spec.groupby('family'):
    g = g.sort_values('graph_size')
    ax.plot(g['spectral_PC1'], g['spectral_PC2'], marker='o', label=family)
    for _, r in g.iterrows():
        ax.text(r['spectral_PC1'], r['spectral_PC2'], f"N={int(r['graph_size'])}", fontsize=8)
ax.axhline(0, linestyle='--', alpha=0.5)
ax.axvline(0, linestyle='--', alpha=0.5)
ax.set_title('Topology-family spectral trajectories')
ax.set_xlabel('spectral PC1')
ax.set_ylabel('spectral PC2')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / '27_topology_family_spectral_trajectories.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print('wrote:', out)


## 8. Figure 6 — Spectral overlap matrix

Family spectra are compared using cosine similarity over graph-level spectral descriptors.
This defines a direct spectral overlap measurement between topology families.


In [ ]:
family_centroids = family_spec.groupby('family')[num_cols].mean()
Xc = StandardScaler().fit_transform(family_centroids.replace([np.inf,-np.inf], np.nan).fillna(0.0))
sim = cosine_similarity(Xc)
# Normalize to 0..1 for display when values include negatives.
sim01 = (sim - sim.min()) / (sim.max() - sim.min() + 1e-12)

sim_df = pd.DataFrame(sim01, index=family_centroids.index, columns=family_centroids.index)
sim_df.to_csv(RESULTS_DIR / '27_spectral_overlap_matrix.csv')

fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(sim01, vmin=0, vmax=1)
ax.set_title('Topology-family spectral overlap matrix')
ax.set_xticks(np.arange(len(sim_df.columns)))
ax.set_xticklabels(sim_df.columns, rotation=45, ha='right')
ax.set_yticks(np.arange(len(sim_df.index)))
ax.set_yticklabels(sim_df.index)
for i in range(sim01.shape[0]):
    for j in range(sim01.shape[1]):
        ax.text(j, i, f'{sim01[i,j]:.2f}', ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, label='normalized spectral overlap')
fig.tight_layout()
out = FIGURES_DIR / '27_spectral_overlap_matrix.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print('wrote:', out)


## 9. Paper-ready spectral summary

This cell writes a short markdown summary that can be adapted into the manuscript.


In [ ]:
summary_md = f'''# Notebook 27 spectral transition summary

Source data: `{source_path}`

## Core spectral quantities

- Residual manifold nodes: {n_nodes}
- kNN graph size: k={k}
- Full spectral entropy: {spectral_entropy:.4f}
- Smallest nontrivial residual Laplacian eigenvalue: {evals[1] if len(evals)>1 else 0.0:.4f}
- Largest residual Laplacian eigenvalue: {evals[-1]:.4f}

## Paper interpretation

The residual Laplacian spectrum provides a true operator-level diagnostic for the residual universality manifold.
Low-frequency modes encode large-scale topology-family separability, while eigenvalue spacing and spectral entropy describe transitions from coarse continuity regions to finer residual variation.
Family-level spectral trajectories extend this structure across graph-size scaling, and spectral overlap measurements quantify topology-family proximity in spectral coordinates.

## Generated paper figures

- `27_residual_laplacian_eigenspectrum.png`
- `27_spectral_density_and_cumulative.png`
- `27_eigenvalue_spacing.png`
- `27_spectral_entropy_curve.png`
- `27_topology_family_spectral_trajectories.png`
- `27_spectral_overlap_matrix.png`
'''

summary_path = EXPORTS_DIR / '27_spectral_transition_summary.md'
summary_path.write_text(summary_md)
print(summary_md)
print('wrote:', summary_path)


## 10. Export bundle

This creates a zip containing all Notebook 27 figures, CSV outputs, and a markdown summary.


In [ ]:
manifest = {
    'notebook': '27_residual_spectral_transition_analysis.ipynb',
    'source_file': str(source_path),
    'created_outputs': {
        'figures': sorted([p.name for p in FIGURES_DIR.glob('27_*.png')]),
        'results': sorted([p.name for p in RESULTS_DIR.glob('27_*')]),
        'exports': sorted([p.name for p in EXPORTS_DIR.glob('27_*')]),
    },
}

manifest_path = EXPORTS_DIR / '27_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / '27_residual_spectral_transition_analysis_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in FIGURES_DIR.glob('27_*.png'):
        z.write(p, arcname=f'figures/{p.name}')
    for p in RESULTS_DIR.glob('27_*'):
        z.write(p, arcname=f'results/{p.name}')
    for p in EXPORTS_DIR.glob('27_*'):
        if p != zip_path:
            z.write(p, arcname=f'exports/{p.name}')

print('Wrote:', zip_path)
print('Zip size MB:', round(zip_path.stat().st_size / 1e6, 3))

# Optional Colab download.
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Colab download skipped. Download manually from:', zip_path)
